In [ ]:
# Cell 1 - Install dependencies
# Run this cell once per kernel if dependencies are missing.
%pip install -q python-dotenv pandas requests tenacity snowflake-connector-python[pandas] ipywidgets


In [ ]:
# Cell 2 - Helpers and configuration
import getpass
import hashlib
import html
import json
import logging
import os
import re
from collections import defaultdict
from datetime import date, datetime, timezone
from decimal import Decimal
from email.utils import parsedate_to_datetime
from pathlib import Path
from typing import Optional
from urllib.parse import parse_qsl, parse_qs, urlencode, urlparse, urlunparse

import pandas as pd
import requests
from dotenv import load_dotenv
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)


NOTEBOOK_FILENAME = "snowflake_trust_center_to_secops.ipynb"
SQL_CHECKS_DIRNAME = "sql_checks"
ACCOUNT_ENV_PREFIXES = (
    "SNOWFLAKE_ACCOUNT_",
    "SNOWFLAKE_USER_",
    "SNOWFLAKE_PRIVATE_KEY_PATH_",
    "SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_",
    "SNOWFLAKE_ROLE_",
    "SNOWFLAKE_WAREHOUSE_",
    "SNOWFLAKE_LABEL_",
)
SECURITY_CHECK_ENV_PREFIXES = (
    "SECURITY_CHECK_NAME_",
    "SECURITY_CHECK_SQL_",
    "SECURITY_CHECK_SQL_FILE_",
)
TIMESTAMP_FIELD_CANDIDATES = (
    "event_timestamp",
    "timestamp",
    "observed_at",
    "query_start_time",
    "start_time",
    "created_on",
    "updated_on",
    "completed_at",
)
DEFAULT_QUERY_TEXT_LIMIT = 10000
DEFAULT_CHRONICLE_PARSER = "SNOWFLAKE"
DEFAULT_CHRONICLE_TRANSPORT = "feeds:importPushLogs"
_MAX_REQUEST_BYTES = (4 * 1024 * 1024) - 1024
logging.basicConfig(level=logging.WARNING)
_log = logging.getLogger("secops_sender")
_DEFAULT_SECOPS_WAIT = wait_exponential(multiplier=1, min=2, max=30)


class RetryableSecOpsError(Exception):
    """Raised for retryable webhook responses."""

    def __init__(self, message: str, retry_after: Optional[int] = None):
        super().__init__(message)
        self.retry_after = retry_after


class NonRetryableSecOpsError(Exception):
    """Raised for permanent webhook failures."""


def detect_project_dir() -> Path:
    """Resolve the notebook directory or fall back to the current working directory."""
    override = os.getenv("NOTEBOOK_ROOT", "").strip()
    if override:
        return Path(override).expanduser().resolve()

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        if (root / NOTEBOOK_FILENAME).exists():
            return root.resolve()
    return Path.cwd().resolve()


def resolve_env_path(project_dir: Optional[Path] = None) -> Path:
    """Resolve the configuration file path."""
    project_dir = project_dir or detect_project_dir()
    env_path = os.getenv("CONFIG_ENV_PATH", str(project_dir / ".env"))
    return Path(env_path).expanduser().resolve()


def load_environment(env_path: Optional[Path] = None) -> tuple[Path, Path]:
    """Load `.env` into the process environment and return the resolved paths."""
    project_dir = detect_project_dir()
    resolved_env_path = Path(env_path or resolve_env_path(project_dir)).expanduser().resolve()
    load_dotenv(resolved_env_path, override=True)
    return project_dir, resolved_env_path


def normalize_private_key_path(raw_path: str) -> str:
    """Normalize the configured private key path for display and connector use."""
    raw_path = (raw_path or "").strip()
    if not raw_path:
        return ""
    return str(Path(raw_path).expanduser().resolve())


def decode_multiline_env_value(raw_value: str) -> str:
    """Turn escaped line breaks from `.env` into real newlines."""
    value = (raw_value or "").strip()
    if not value:
        return ""
    return value.replace("\\n", "\n").replace("\\t", "\t")


def collect_suffix_indexes(environ: dict, prefixes: tuple[str, ...]) -> list[int]:
    """Collect numeric suffixes for the provided environment variable prefixes."""
    indexes = set()
    for key in environ:
        for prefix in prefixes:
            if key.startswith(prefix):
                suffix = key[len(prefix) :]
                if suffix.isdigit():
                    indexes.add(int(suffix))
    return sorted(indexes)


def normalize_check_key(raw_value: str, default: str) -> str:
    """Build a stable slug-style key for one security check."""
    text = re.sub(r"[^a-z0-9]+", "_", (raw_value or "").strip().lower()).strip("_")
    return text or default


def parse_csv_values(raw_value: str) -> list[str]:
    """Parse a comma-separated metadata field."""
    return [item.strip() for item in (raw_value or "").split(",") if item.strip()]


def resolve_sql_file_path(raw_path: str, project_dir: Path) -> Path:
    """Resolve a SQL file path relative to the project directory."""
    path = Path(raw_path).expanduser()
    if not path.is_absolute():
        path = (project_dir / path).resolve()
    else:
        path = path.resolve()
    return path


def parse_sql_file(sql_path: Path) -> tuple[dict, str]:
    """Parse top-of-file metadata comments and return metadata plus SQL body."""
    if not sql_path.exists():
        raise FileNotFoundError(f"SQL file not found: {sql_path}")

    content = sql_path.read_text(encoding="utf-8")
    metadata = {}
    sql_lines = []
    in_header = True
    for line in content.splitlines():
        stripped = line.strip()
        if in_header and (not stripped):
            continue
        if in_header and stripped.startswith("--"):
            match = re.match(r"^--\s*([a-z0-9_]+)\s*:\s*(.*?)\s*$", stripped, re.IGNORECASE)
            if match:
                metadata[match.group(1).lower()] = match.group(2).strip()
            continue
        in_header = False
        sql_lines.append(line)

    sql = "\n".join(sql_lines).strip()
    if not sql:
        raise ValueError(f"SQL file is empty after metadata header: {sql_path}")
    return metadata, sql


def build_accounts(environ: dict, prompt_for_missing: bool = True) -> list[dict]:
    """Build Snowflake account configuration from environment variables."""
    indexes = collect_suffix_indexes(environ, ACCOUNT_ENV_PREFIXES)
    accounts = []
    for idx in indexes:
        account = environ.get(f"SNOWFLAKE_ACCOUNT_{idx}", "").strip()
        user = environ.get(f"SNOWFLAKE_USER_{idx}", "").strip()
        private_key_path = normalize_private_key_path(
            environ.get(f"SNOWFLAKE_PRIVATE_KEY_PATH_{idx}", "")
        )
        private_key_passphrase = environ.get(f"SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_{idx}", "")
        role = environ.get(f"SNOWFLAKE_ROLE_{idx}", "").strip()
        warehouse = environ.get(f"SNOWFLAKE_WAREHOUSE_{idx}", "").strip()
        label = environ.get(f"SNOWFLAKE_LABEL_{idx}", "").strip() or f"account_{idx}"

        if not account:
            continue

        missing = []
        if not user:
            missing.append(f"SNOWFLAKE_USER_{idx}")
        if not private_key_path:
            missing.append(f"SNOWFLAKE_PRIVATE_KEY_PATH_{idx}")
        if missing:
            raise ValueError(
                f"Snowflake account {idx} is missing required values: {', '.join(missing)}"
            )

        accounts.append(
            {
                "index": idx,
                "key": f"account_{idx}",
                "label": label,
                "account": account,
                "user": user,
                "private_key_path": private_key_path,
                "private_key_passphrase": private_key_passphrase,
                "role": role,
                "warehouse": warehouse,
            }
        )

    if accounts:
        return accounts
    if not prompt_for_missing:
        raise ValueError("At least one Snowflake account is required.")

    account = input("Snowflake account locator: ").strip()
    user = input("Snowflake user: ").strip()
    private_key_path = normalize_private_key_path(input("Snowflake private key path: ").strip())
    private_key_passphrase = getpass.getpass(
        "Snowflake private key passphrase (leave blank if not encrypted): "
    )
    if not account or not user or not private_key_path:
        raise ValueError("Snowflake account, user, and private key path are required.")

    return [
        {
            "index": 1,
            "key": "account_1",
            "label": "account_1",
            "account": account,
            "user": user,
            "private_key_path": private_key_path,
            "private_key_passphrase": private_key_passphrase,
            "role": "",
            "warehouse": "",
        }
    ]


def build_security_checks(environ: dict, project_dir: Path) -> list[dict]:
    """Build configured SQL checks from environment variables and SQL files."""
    indexes = collect_suffix_indexes(environ, SECURITY_CHECK_ENV_PREFIXES)
    checks = []
    seen_keys = set()

    for idx in indexes:
        env_name = environ.get(f"SECURITY_CHECK_NAME_{idx}", "").strip()
        raw_sql = environ.get(f"SECURITY_CHECK_SQL_{idx}", "")
        raw_sql_file = environ.get(f"SECURITY_CHECK_SQL_FILE_{idx}", "").strip()
        metadata = {}
        sql = ""
        sql_path = None

        if raw_sql_file:
            sql_path = resolve_sql_file_path(raw_sql_file, project_dir)
            metadata, sql = parse_sql_file(sql_path)
        else:
            sql = decode_multiline_env_value(raw_sql)

        if not sql:
            if env_name or raw_sql_file:
                raise ValueError(
                    f"Security check {idx} is missing SQL. Configure SECURITY_CHECK_SQL_FILE_{idx} or SECURITY_CHECK_SQL_{idx}."
                )
            continue

        default_key = normalize_check_key(
            env_name or metadata.get("name") or (sql_path.stem if sql_path else ""),
            f"security_check_{idx}",
        )
        if default_key in seen_keys:
            raise ValueError(f"Duplicate security check key detected: {default_key}")
        seen_keys.add(default_key)

        title = metadata.get("title") or default_key.replace("_", " ").title()
        domain = metadata.get("domain") or "Custom"
        priority = metadata.get("priority") or "P2"
        sources = parse_csv_values(metadata.get("sources", ""))
        watermark_column = metadata.get("watermark_column", "")
        timestamp_field = metadata.get("timestamp_field") or watermark_column
        native_id_field = metadata.get("native_id_field", "")
        max_latency_minutes = int(metadata.get("max_latency_minutes", "0") or 0)
        recommended_cadence = metadata.get("recommended_cadence") or "manual"
        chronicle_log_type = metadata.get("chronicle_log_type") or "custom_query"
        description = metadata.get("description") or "Custom Snowflake security check."

        checks.append(
            {
                "index": idx,
                "key": default_key,
                "name": env_name or default_key,
                "title": title,
                "domain": domain,
                "priority": priority,
                "sources": sources,
                "watermark_column": watermark_column,
                "timestamp_field": timestamp_field,
                "native_id_field": native_id_field,
                "max_latency_minutes": max_latency_minutes,
                "recommended_cadence": recommended_cadence,
                "chronicle_log_type": chronicle_log_type,
                "description": description,
                "parser_mode": "chronicle_native_snowflake",
                "sql": sql,
                "sql_path": str(sql_path) if sql_path else "",
            }
        )

    if not checks:
        raise ValueError("At least one security check is required.")
    return checks


def url_has_query_credential(url: str, key: str) -> bool:
    """Return True when the webhook URL already carries the requested auth parameter."""
    values = parse_qs(urlparse(url).query).get(key, [])
    return any(value.strip() for value in values)


def describe_auth_mode(url: str) -> str:
    """Describe how the webhook is authenticated."""
    has_key = url_has_query_credential(url, "key")
    has_secret = url_has_query_credential(url, "secret")
    if has_key and has_secret:
        return "url_query"
    if (not has_key) and (not has_secret):
        return "headers"
    return "mixed"


def mask_webhook_url(url: str) -> str:
    """Mask key and secret values when rendering the webhook URL."""
    if not url:
        return "<unset>"
    parsed = urlparse(url)
    masked_items = []
    for item_key, item_value in parse_qsl(parsed.query, keep_blank_values=True):
        if item_key in {"key", "secret"} and item_value:
            masked_items.append((item_key, "***"))
        else:
            masked_items.append((item_key, item_value))
    return urlunparse(parsed._replace(query=urlencode(masked_items)))


def build_runtime_config(
    environ: dict,
    project_dir: Path,
    env_path: Path,
    prompt_for_missing: bool = True,
) -> dict:
    """Build the notebook runtime configuration."""
    accounts = build_accounts(environ, prompt_for_missing=prompt_for_missing)
    security_checks = build_security_checks(environ, project_dir=project_dir)

    webhook_url = environ.get("SECOPS_WEBHOOK_URL", "").strip()
    if not webhook_url and prompt_for_missing:
        webhook_url = input("Google SecOps webhook URL: ").strip()
    if not webhook_url:
        raise ValueError("SECOPS_WEBHOOK_URL is required.")

    config = {
        "PROJECT_DIR": str(project_dir),
        "ENV_PATH": str(env_path),
        "ENV_FOUND": env_path.exists(),
        "CHRONICLE_PARSER": DEFAULT_CHRONICLE_PARSER,
        "CHRONICLE_TRANSPORT": DEFAULT_CHRONICLE_TRANSPORT,
        "QUERY_TEXT_MAX_CHARS": DEFAULT_QUERY_TEXT_LIMIT,
        "SNOWFLAKE_ACCOUNTS": accounts,
        "SECURITY_CHECKS": security_checks,
        "SECOPS_WEBHOOK_URL": webhook_url,
        "SECOPS_API_KEY": environ.get("SECOPS_API_KEY", "").strip(),
        "SECOPS_WEBHOOK_SECRET": environ.get("SECOPS_WEBHOOK_SECRET", "").strip(),
        "BATCH_SIZE": int(environ.get("BATCH_SIZE", "100")),
        "DRY_RUN": environ.get("DRY_RUN", "false").strip().lower() == "true",
        "QUERY_RESULTS": {},
        "SEND_SELECTION": [],
    }

    if not config["DRY_RUN"] and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if prompt_for_missing and not config["SECOPS_API_KEY"]:
            config["SECOPS_API_KEY"] = getpass.getpass("Google SecOps API key: ")
    if not config["DRY_RUN"] and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if prompt_for_missing and not config["SECOPS_WEBHOOK_SECRET"]:
            config["SECOPS_WEBHOOK_SECRET"] = getpass.getpass("Google SecOps webhook secret: ")

    missing_auth = []
    if (
        not config["DRY_RUN"]
        and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key")
        and not config["SECOPS_API_KEY"]
    ):
        missing_auth.append("SECOPS_API_KEY")
    if (
        not config["DRY_RUN"]
        and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret")
        and not config["SECOPS_WEBHOOK_SECRET"]
    ):
        missing_auth.append("SECOPS_WEBHOOK_SECRET")
    if missing_auth:
        raise ValueError("Missing Google SecOps auth values: " + ", ".join(missing_auth))

    return config


def summarize_runtime_config(config: dict) -> dict:
    """Build a concise configuration summary for notebook output."""
    return {
        "project_dir": config["PROJECT_DIR"],
        "env_path": config["ENV_PATH"],
        "env_found": config["ENV_FOUND"],
        "chronicle_parser": config["CHRONICLE_PARSER"],
        "chronicle_transport": config["CHRONICLE_TRANSPORT"],
        "accounts": [
            {
                "label": account["label"],
                "account": account["account"],
                "user": account["user"],
                "role": account["role"] or "<default>",
                "warehouse": account["warehouse"] or "<default>",
            }
            for account in config["SNOWFLAKE_ACCOUNTS"]
        ],
        "security_checks": [
            {
                "key": check["key"],
                "title": check["title"],
                "domain": check["domain"],
                "priority": check["priority"],
                "sources": check["sources"],
                "watermark_column": check["watermark_column"],
                "max_latency_minutes": check["max_latency_minutes"],
                "recommended_cadence": check["recommended_cadence"],
                "chronicle_log_type": check["chronicle_log_type"],
                "sql_path": check["sql_path"],
            }
            for check in config["SECURITY_CHECKS"]
        ],
        "batch_size": config["BATCH_SIZE"],
        "dry_run_default": config["DRY_RUN"],
        "secops_auth_mode": describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
        "secops_webhook_url": mask_webhook_url(config["SECOPS_WEBHOOK_URL"]),
    }


def make_group_key(account_label: str, query_key: str) -> str:
    """Build a stable key for one account/query group."""
    return f"{account_label}::{query_key}"


def split_group_key(group_key: str) -> tuple[str, str]:
    """Split one account/query group key."""
    account_label, query_key = group_key.split("::", 1)
    return account_label, query_key


def build_domain_catalog(config: dict) -> list[dict]:
    """Group checks by domain for the analyst-facing UI."""
    buckets = defaultdict(list)
    for check in config["SECURITY_CHECKS"]:
        buckets[check["domain"]].append(check)
    return [
        {"domain": domain, "checks": checks}
        for domain, checks in buckets.items()
    ]


def build_group_catalog(config: dict) -> dict:
    """Build the account/check matrix metadata."""
    check_lookup = {check["key"]: check for check in config["SECURITY_CHECKS"]}
    account_lookup = {account["label"]: account for account in config["SNOWFLAKE_ACCOUNTS"]}
    group_lookup = {}
    group_order = []

    for account in config["SNOWFLAKE_ACCOUNTS"]:
        for check in config["SECURITY_CHECKS"]:
            group_key = make_group_key(account["label"], check["key"])
            group_meta = {
                "group_key": group_key,
                "account_label": account["label"],
                "query_key": check["key"],
                "query_title": check["title"],
                "account": account,
                "query": check,
            }
            group_lookup[group_key] = group_meta
            group_order.append(group_key)

    return {
        "accounts": config["SNOWFLAKE_ACCOUNTS"],
        "queries": config["SECURITY_CHECKS"],
        "account_lookup": account_lookup,
        "check_lookup": check_lookup,
        "domains": build_domain_catalog(config),
        "group_lookup": group_lookup,
        "group_order": group_order,
    }


def normalize_group_keys(group_catalog: dict, group_keys: list[str]) -> list[str]:
    """Normalize group selection order and validate keys."""
    requested = list(group_keys or [])
    invalid = sorted(set(requested) - set(group_catalog["group_lookup"]))
    if invalid:
        raise ValueError(f"Unknown group keys: {invalid}")
    requested_set = set(requested)
    return [group_key for group_key in group_catalog["group_order"] if group_key in requested_set]


def build_default_group_selection_state(group_catalog: dict) -> dict:
    """Enable every check for every account by default; account selection limits visibility."""
    return {group_key: True for group_key in group_catalog["group_order"]}


def get_default_account_labels(config: dict) -> list[str]:
    """Return the default selected account list."""
    if not config["SNOWFLAKE_ACCOUNTS"]:
        return []
    return [config["SNOWFLAKE_ACCOUNTS"][0]["label"]]


def compute_checked_groups(
    group_catalog: dict,
    group_selection_state: dict,
    selected_accounts: list[str],
) -> list[str]:
    """Build the active account/check selection from per-group state and visible accounts."""
    selected_account_set = set(selected_accounts)
    return [
        group_key
        for group_key in group_catalog["group_order"]
        if split_group_key(group_key)[0] in selected_account_set
        and bool(group_selection_state.get(group_key, False))
    ]


def build_empty_query_result(group_meta: dict) -> dict:
    """Create the default result object for one account/check row."""
    return {
        "group_key": group_meta["group_key"],
        "account_label": group_meta["account_label"],
        "account_locator": group_meta["account"]["account"],
        "query_key": group_meta["query_key"],
        "query_name": group_meta["query"]["name"],
        "query_title": group_meta["query"]["title"],
        "domain": group_meta["query"]["domain"],
        "priority": group_meta["query"]["priority"],
        "sources": list(group_meta["query"]["sources"]),
        "watermark_column": group_meta["query"]["watermark_column"],
        "timestamp_field": group_meta["query"]["timestamp_field"],
        "native_id_field": group_meta["query"]["native_id_field"],
        "max_latency_minutes": group_meta["query"]["max_latency_minutes"],
        "recommended_cadence": group_meta["query"]["recommended_cadence"],
        "chronicle_log_type": group_meta["query"]["chronicle_log_type"],
        "description": group_meta["query"]["description"],
        "status": "pending",
        "row_count": 0,
        "columns": [],
        "dataframe": pd.DataFrame(),
        "error": None,
    }


def build_initial_query_results(config: dict, group_catalog: Optional[dict] = None) -> dict:
    """Build an empty result object for every account/check row."""
    group_catalog = group_catalog or build_group_catalog(config)
    return {
        group_key: build_empty_query_result(group_meta)
        for group_key, group_meta in group_catalog["group_lookup"].items()
    }


def get_selection_snapshot(
    checked_groups: list[str],
    dry_run: bool,
    group_catalog: dict,
) -> dict:
    """Build a normalized runtime selection snapshot."""
    return {
        "checked_groups": normalize_group_keys(group_catalog, checked_groups),
        "dry_run": bool(dry_run),
    }


def sync_config_runtime_fields(config: dict, app_state: dict) -> None:
    """Mirror runtime state onto CONFIG fields used elsewhere in the notebook."""
    config["QUERY_RESULTS"] = app_state["query_results"]
    config["SEND_SELECTION"] = list(app_state["checked_groups"])


def create_app_state(config: dict) -> dict:
    """Create notebook runtime state."""
    group_catalog = build_group_catalog(config)
    selected_accounts = get_default_account_labels(config)
    group_selection_state = build_default_group_selection_state(group_catalog)
    checked_groups = compute_checked_groups(group_catalog, group_selection_state, selected_accounts)
    app_state = {
        "group_catalog": group_catalog,
        "selected_accounts": selected_accounts,
        "selected_checks": [check["key"] for check in config["SECURITY_CHECKS"]],
        "group_selection_state": group_selection_state,
        "checked_groups": checked_groups,
        "dry_run": config["DRY_RUN"],
        "last_run_selection": None,
        "selection_dirty": False,
        "dirty_reason": "Run selected groups first.",
        "sanity_check_results": [],
        "query_results": build_initial_query_results(config, group_catalog),
        "payload_plan": {
            "selected_groups": [],
            "total_events": 0,
            "generated_at": None,
            "dry_run": config["DRY_RUN"],
        },
        "selected_events": [],
        "secops_result": None,
    }
    sync_config_runtime_fields(config, app_state)
    return app_state


def set_selected_accounts(app_state: dict, account_labels: list[str]) -> None:
    """Update the visible account set and recompute checked groups."""
    validate_account_labels_from_catalog(app_state["group_catalog"], account_labels)
    app_state["selected_accounts"] = list(account_labels)
    app_state["checked_groups"] = compute_checked_groups(
        app_state["group_catalog"],
        app_state["group_selection_state"],
        app_state["selected_accounts"],
    )


def set_check_selection_state(app_state: dict, query_key: str, selected: bool) -> None:
    """Apply a master check selection across all accounts."""
    for group_key in app_state["group_catalog"]["group_order"]:
        _account_label, current_query_key = split_group_key(group_key)
        if current_query_key == query_key:
            app_state["group_selection_state"][group_key] = bool(selected)
    app_state["selected_checks"] = [
        check["key"]
        for check in app_state["group_catalog"]["queries"]
        if any(
            app_state["group_selection_state"].get(make_group_key(account["label"], check["key"]), False)
            for account in app_state["group_catalog"]["accounts"]
        )
    ]
    app_state["checked_groups"] = compute_checked_groups(
        app_state["group_catalog"],
        app_state["group_selection_state"],
        app_state["selected_accounts"],
    )


def set_group_selection_state(app_state: dict, group_key: str, selected: bool) -> None:
    """Update one account/check group selection."""
    app_state["group_selection_state"][group_key] = bool(selected)
    app_state["checked_groups"] = compute_checked_groups(
        app_state["group_catalog"],
        app_state["group_selection_state"],
        app_state["selected_accounts"],
    )


def sync_runtime_selection(app_state: dict, checked_groups: list[str], dry_run: bool) -> dict:
    """Update runtime selection and compute whether a rerun is required."""
    snapshot = get_selection_snapshot(checked_groups, dry_run, app_state["group_catalog"])
    previous_run = app_state.get("last_run_selection")

    app_state["checked_groups"] = snapshot["checked_groups"]
    app_state["dry_run"] = snapshot["dry_run"]

    if previous_run is None:
        app_state["selection_dirty"] = False
        app_state["dirty_reason"] = "Run selected groups first."
        return snapshot

    selection_changed = previous_run["checked_groups"] != snapshot["checked_groups"]
    mode_changed = previous_run["dry_run"] != snapshot["dry_run"]
    if selection_changed or mode_changed:
        app_state["selection_dirty"] = True
        if selection_changed and mode_changed:
            app_state["dirty_reason"] = "Selection changed and mode changed; rerun required before send."
        elif selection_changed:
            app_state["dirty_reason"] = "Selection changed; rerun required before send."
        else:
            app_state["dirty_reason"] = "Mode changed; rerun required before send."
    else:
        app_state["selection_dirty"] = False
        app_state["dirty_reason"] = "Ready to send."
    return snapshot


def validate_account_labels_from_catalog(group_catalog: dict, account_labels: list[str]) -> None:
    """Validate account labels against the group catalog."""
    available_accounts = {account["label"] for account in group_catalog["accounts"]}
    invalid_accounts = sorted(set(account_labels) - available_accounts)
    if invalid_accounts:
        raise ValueError(f"Unknown Snowflake account labels: {invalid_accounts}")


def build_run_table_rows(config: dict, app_state: dict) -> list[dict]:
    """Build the analyst-facing run table rows."""
    rows = []
    selected_account_set = set(app_state["selected_accounts"])
    for account in config["SNOWFLAKE_ACCOUNTS"]:
        if account["label"] not in selected_account_set:
            continue
        for check in config["SECURITY_CHECKS"]:
            group_key = make_group_key(account["label"], check["key"])
            result = app_state["query_results"][group_key]
            rows.append(
                {
                    "group_key": group_key,
                    "selected": bool(app_state["group_selection_state"].get(group_key, False)),
                    "account": account["label"],
                    "check": check["title"],
                    "domain": check["domain"],
                    "source": ", ".join(check["sources"]),
                    "watermark": check["watermark_column"],
                    "latency": check["max_latency_minutes"],
                    "cadence": check["recommended_cadence"],
                    "log_type": check["chronicle_log_type"],
                    "status": result["status"],
                    "row_count": result["row_count"],
                    "error": result["error"],
                }
            )
    return rows


def reduce_send_state(app_state: dict) -> dict:
    """Determine whether the current runtime selection can be sent to SecOps."""
    eligible_groups = []
    for group_key in app_state["checked_groups"]:
        result = app_state["query_results"].get(group_key)
        if result and result["status"] == "success" and result["row_count"] > 0:
            eligible_groups.append(group_key)

    if not app_state["checked_groups"]:
        return {
            "enabled": False,
            "reason": "Select at least one account/check row.",
            "eligible_groups": eligible_groups,
        }
    if app_state["last_run_selection"] is None:
        return {
            "enabled": False,
            "reason": "Run selected rows first.",
            "eligible_groups": eligible_groups,
        }
    if app_state["selection_dirty"]:
        return {
            "enabled": False,
            "reason": app_state["dirty_reason"],
            "eligible_groups": eligible_groups,
        }
    if not eligible_groups:
        return {
            "enabled": False,
            "reason": "No selected rows have successful non-empty results.",
            "eligible_groups": eligible_groups,
        }
    return {
        "enabled": True,
        "reason": f"{len(eligible_groups)} selected row(s) ready to send.",
        "eligible_groups": eligible_groups,
    }


def summarize_query_results(query_results: dict) -> dict:
    """Summarize query execution outcomes."""
    summary = {
        "groups": len(query_results),
        "pending": 0,
        "running": 0,
        "success": 0,
        "empty": 0,
        "error": 0,
        "rows": 0,
    }
    for result in query_results.values():
        status = result["status"]
        if status in summary:
            summary[status] += 1
        summary["rows"] += result.get("row_count", 0)
    return summary


def build_header_snapshot(config: dict, app_state: dict) -> dict:
    """Build the compact control-panel header state."""
    send_state = reduce_send_state(app_state)
    return {
        "selected_account_count": len(app_state["selected_accounts"]),
        "visible_group_count": len(build_run_table_rows(config, app_state)),
        "selected_group_count": len(app_state["checked_groups"]),
        "eligible_group_count": len(send_state["eligible_groups"]),
        "dry_run": app_state["dry_run"],
        "auth_mode": describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
        "masked_webhook_url": mask_webhook_url(config["SECOPS_WEBHOOK_URL"]),
        "chronicle_parser": config["CHRONICLE_PARSER"],
        "chronicle_transport": config["CHRONICLE_TRANSPORT"],
        "selection_dirty": app_state["selection_dirty"],
        "dirty_reason": app_state["dirty_reason"],
        "send_enabled": send_state["enabled"],
        "send_reason": send_state["reason"],
    }


def build_status_badge(result: dict) -> str:
    """Render one small HTML status badge for the run table."""
    status = result.get("status", "pending")
    if status == "success":
        label = f"ok:{result.get('row_count', 0)}"
        color = "#0f766e"
        accent = "#ccfbf1"
        detail = f"{result.get('row_count', 0)} row(s)"
    elif status == "empty":
        label = "empty"
        color = "#92400e"
        accent = "#fef3c7"
        detail = "0 row(s)"
    elif status == "error":
        label = "error"
        color = "#b91c1c"
        accent = "#fee2e2"
        detail = html.escape((result.get("error") or "")[:120])
    elif status == "running":
        label = "running"
        color = "#1d4ed8"
        accent = "#dbeafe"
        detail = "Executing"
    else:
        label = "pending"
        color = "#475569"
        accent = "#e2e8f0"
        detail = "Waiting"

    return (
        f"<div style='display:inline-block;padding:2px 8px;border-radius:999px;"
        f"background:{accent};color:{color};font-size:11px;font-weight:600'>{label}</div>"
        f"<div style='margin-top:4px;font-size:11px;color:#475569'>{detail}</div>"
    )


def connect_snowflake(acct_cfg: dict):
    """Create a Snowflake connection using key pair authentication."""
    from pathlib import Path as _Path

    import snowflake.connector

    private_key_path = _Path(acct_cfg["private_key_path"]).expanduser().resolve()
    if not private_key_path.exists():
        raise FileNotFoundError(
            f"[{acct_cfg['label']}] Private key file not found: {private_key_path}"
        )

    connect_kwargs = {
        "account": acct_cfg["account"],
        "user": acct_cfg["user"],
        "authenticator": "SNOWFLAKE_JWT",
        "private_key_file": str(private_key_path),
        "private_key_file_pwd": acct_cfg.get("private_key_passphrase") or None,
        "client_session_keep_alive": False,
        "network_timeout": 30,
    }
    if acct_cfg.get("role"):
        connect_kwargs["role"] = acct_cfg["role"]
    if acct_cfg.get("warehouse"):
        connect_kwargs["warehouse"] = acct_cfg["warehouse"]
    return snowflake.connector.connect(**connect_kwargs)


def close_connection(conn) -> None:
    """Close a Snowflake connection when possible."""
    if conn is None:
        return
    close = getattr(conn, "close", None)
    if callable(close):
        close()


def fetch_query_dataframe(conn, sql: str) -> pd.DataFrame:
    """Execute one SQL statement and return the result as a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if not cur.description:
            return pd.DataFrame()
        columns = [column[0] for column in cur.description]
        frames = []
        try:
            for batch in cur.fetch_pandas_batches():
                frames.append(batch.reindex(columns=columns))
        except Exception:
            rows = cur.fetchall()
            frames = [pd.DataFrame(rows, columns=columns)]
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True).reindex(columns=columns)


def run_selected_account_sanity_checks(
    config: dict,
    account_labels: list[str],
    connect_fn=None,
    close_fn=None,
) -> list[dict]:
    """Run a basic session query for the selected accounts."""
    validate_account_labels_from_catalog(build_group_catalog(config), account_labels)
    account_lookup = {account["label"]: account for account in config["SNOWFLAKE_ACCOUNTS"]}
    connect_fn = connect_fn or connect_snowflake
    close_fn = close_fn or close_connection
    results = []
    for account_label in account_labels:
        conn = None
        try:
            conn = connect_fn(account_lookup[account_label])
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT CURRENT_ACCOUNT(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()"
                )
                current_account, current_role, current_warehouse, current_version = cur.fetchone()
            results.append(
                {
                    "account_label": account_label,
                    "status": "success",
                    "current_account": current_account,
                    "current_role": current_role,
                    "current_warehouse": current_warehouse,
                    "current_version": current_version,
                }
            )
        except Exception as exc:
            results.append(
                {
                    "account_label": account_label,
                    "status": "error",
                    "error": str(exc),
                }
            )
        finally:
            close_fn(conn)
    return results


def execute_selected_groups(
    config: dict,
    selected_group_keys: list[str],
    connect_fn=None,
    fetch_fn=None,
    close_fn=None,
    progress_fn=None,
) -> dict:
    """Execute the selected account/check rows."""
    group_catalog = build_group_catalog(config)
    selected_group_keys = normalize_group_keys(group_catalog, selected_group_keys)
    connect_fn = connect_fn or connect_snowflake
    fetch_fn = fetch_fn or fetch_query_dataframe
    close_fn = close_fn or close_connection

    results = {}
    for group_key in selected_group_keys:
        group_meta = group_catalog["group_lookup"][group_key]
        result = build_empty_query_result(group_meta)
        result["status"] = "running"
        if progress_fn:
            progress_fn(group_key, result.copy())

        conn = None
        try:
            conn = connect_fn(group_meta["account"])
            dataframe = fetch_fn(conn, group_meta["query"]["sql"]).copy()
            result["dataframe"] = dataframe
            result["columns"] = dataframe.columns.tolist()
            result["row_count"] = len(dataframe)
            result["status"] = "empty" if dataframe.empty else "success"
        except Exception as exc:
            result["status"] = "error"
            result["error"] = str(exc)
        finally:
            close_fn(conn)

        results[group_key] = result
        if progress_fn:
            progress_fn(group_key, result.copy())
    return results


def json_safe_value(value):
    """Convert notebook values into JSON-safe primitives."""
    if value is None or value is pd.NaT:
        return None
    try:
        if pd.isna(value) and not isinstance(value, (dict, list, tuple, set)):
            return None
    except Exception:
        pass
    if isinstance(value, pd.Timestamp):
        if value.tzinfo is None:
            value = value.tz_localize(timezone.utc)
        return value.tz_convert(timezone.utc).isoformat()
    if isinstance(value, datetime):
        if value.tzinfo is None:
            value = value.replace(tzinfo=timezone.utc)
        return value.astimezone(timezone.utc).isoformat()
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, Decimal):
        if value == value.to_integral():
            return int(value)
        return float(value)
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, dict):
        return {str(key): json_safe_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe_value(item) for item in value]
    item = getattr(value, "item", None)
    if callable(item):
        try:
            unpacked = item()
        except Exception:
            unpacked = value
        if unpacked is not value:
            return json_safe_value(unpacked)
    return value


def normalize_field_name(key) -> str:
    """Normalize event field names to parser-friendly snake_case."""
    text = re.sub(r"[^a-z0-9]+", "_", str(key).strip().lower()).strip("_")
    return text or "field"


def extract_source_timestamp(record: dict) -> Optional[str]:
    """Extract a source timestamp from a result record when one is available."""
    lower_map = {normalize_field_name(key): key for key in record}
    for candidate in TIMESTAMP_FIELD_CANDIDATES:
        original_key = lower_map.get(candidate)
        if original_key is None:
            continue
        parsed = pd.to_datetime(record.get(original_key), utc=True, errors="coerce")
        if pd.isna(parsed):
            continue
        return parsed.isoformat()
    return None


def prepare_query_text(value: Optional[str], max_chars: int) -> tuple[Optional[str], Optional[str], bool]:
    """Truncate query text and return the sha256 hash for correlation."""
    if value is None:
        return None, None, False
    text = str(value)
    digest = hashlib.sha256(text.encode("utf-8", errors="replace")).hexdigest()
    truncated = len(text) > max_chars
    if truncated:
        text = text[:max_chars] + " ... <truncated>"
    return text, digest, truncated


def build_stable_record_uid(event: dict, query_result: dict, row_index: int) -> str:
    """Build a stable dedup key for one emitted event."""
    parts = [
        query_result["chronicle_log_type"],
        query_result["account_label"],
    ]
    native_id_field = normalize_field_name(query_result.get("native_id_field", ""))
    if native_id_field and native_id_field != "none" and event.get(native_id_field):
        parts.append(str(event[native_id_field]))
    elif event.get("query_id"):
        parts.append(str(event["query_id"]))
    elif event.get("event_id"):
        parts.append(str(event["event_id"]))
    elif event.get("event_timestamp"):
        parts.append(str(event["event_timestamp"]))
        parts.append(str(row_index))
    else:
        parts.append(str(row_index))
    return ":".join(parts)


def build_parser_friendly_event(
    record: dict,
    query_result: dict,
    generated_at: Optional[str] = None,
    row_index: int = 1,
    query_text_limit: int = DEFAULT_QUERY_TEXT_LIMIT,
) -> dict:
    """Convert one query result row into a Chronicle parser-friendly Snowflake log."""
    generated_at = generated_at or datetime.now(timezone.utc).isoformat()
    normalized_record = {
        normalize_field_name(key): json_safe_value(value)
        for key, value in record.items()
    }

    if "query_text" in normalized_record:
        query_text, query_hash, truncated = prepare_query_text(
            normalized_record.get("query_text"),
            query_text_limit,
        )
        normalized_record["query_text"] = query_text
        if query_hash:
            normalized_record["query_text_sha256"] = query_hash
            normalized_record["query_text_truncated"] = truncated

    event = dict(normalized_record)
    event["application"] = "snowflake"
    event["environment"] = query_result["account_label"]
    event["log_type"] = query_result["chronicle_log_type"]
    event["ingest_time"] = generated_at
    event["snowflake_account"] = query_result["account_label"]
    event["snowflake_account_locator"] = query_result["account_locator"]

    timestamp_field = normalize_field_name(
        query_result.get("timestamp_field") or query_result.get("watermark_column") or ""
    )
    if timestamp_field and event.get(timestamp_field) is not None:
        event.setdefault("event_timestamp", event[timestamp_field])
    if event.get("event_timestamp") is None:
        source_timestamp = extract_source_timestamp(normalized_record)
        if source_timestamp:
            event["event_timestamp"] = source_timestamp

    event["record_uid"] = build_stable_record_uid(event, query_result, row_index)
    return event


def build_group_events(query_result: dict, generated_at: Optional[str] = None) -> list[dict]:
    """Convert one query result group into parser-friendly Snowflake log objects."""
    dataframe = query_result["dataframe"]
    if dataframe.empty:
        return []
    generated_at = generated_at or datetime.now(timezone.utc).isoformat()
    events = []
    for row_index, record in enumerate(dataframe.to_dict(orient="records"), start=1):
        events.append(
            build_parser_friendly_event(
                record,
                query_result=query_result,
                generated_at=generated_at,
                row_index=row_index,
            )
        )
    return events


def build_selected_events(
    query_results: dict,
    selected_group_keys: list[str],
    dry_run: Optional[bool] = None,
) -> tuple[list[dict], dict]:
    """Build payload events for the selected account/check groups."""
    generated_at = datetime.now(timezone.utc).isoformat()
    events = []
    group_summaries = []
    for group_key in selected_group_keys:
        result = query_results.get(group_key)
        if not result or result["status"] != "success" or result["row_count"] == 0:
            continue
        group_events = build_group_events(result, generated_at=generated_at)
        events.extend(group_events)
        group_summaries.append(
            {
                "group_key": group_key,
                "account_label": result["account_label"],
                "query_key": result["query_key"],
                "query_title": result["query_title"],
                "domain": result["domain"],
                "log_type": result["chronicle_log_type"],
                "row_count": len(group_events),
            }
        )
    return events, {
        "selected_groups": group_summaries,
        "total_events": len(events),
        "generated_at": generated_at,
        "dry_run": dry_run,
    }


def build_sender_config(config: dict, app_state: dict) -> dict:
    """Build the runtime sender config using the UI dry-run toggle."""
    sender_config = dict(config)
    sender_config["DRY_RUN"] = bool(app_state["dry_run"])
    return sender_config


def _parse_retry_after(value: Optional[str]) -> Optional[int]:
    """Parse the HTTP Retry-After header."""
    if not value:
        return None
    try:
        return max(0, int(value))
    except ValueError:
        pass
    try:
        retry_at = parsedate_to_datetime(value)
        if retry_at.tzinfo is None:
            retry_at = retry_at.replace(tzinfo=timezone.utc)
        return max(0, int((retry_at - datetime.now(timezone.utc)).total_seconds()))
    except Exception:
        return None


def _secops_wait(retry_state) -> float:
    """Use Retry-After when it is available."""
    exc = retry_state.outcome.exception() if retry_state.outcome else None
    if isinstance(exc, RetryableSecOpsError) and exc.retry_after is not None:
        return exc.retry_after
    return _DEFAULT_SECOPS_WAIT(retry_state)


def _build_headers(config: dict) -> dict:
    """Build SecOps webhook headers."""
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "snowflake-chronicle-standalone-notebook/1.0",
    }
    if not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if not config.get("SECOPS_API_KEY"):
            raise NonRetryableSecOpsError(
                "SECOPS_API_KEY is required because the webhook URL does not include key=."
            )
        headers["X-goog-api-key"] = config["SECOPS_API_KEY"]
    if not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if not config.get("SECOPS_WEBHOOK_SECRET"):
            raise NonRetryableSecOpsError(
                "SECOPS_WEBHOOK_SECRET is required because the webhook URL does not include secret=."
            )
        headers["X-Webhook-Access-Key"] = config["SECOPS_WEBHOOK_SECRET"]
    return headers


def _serialize_event(event: dict) -> bytes:
    """Serialize one event as JSON bytes."""
    return json.dumps(event, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def _build_event_batches(events: list[dict], batch_size: int) -> list[dict]:
    """Split events by count and payload size."""
    if batch_size < 1:
        raise ValueError("BATCH_SIZE must be >= 1.")

    batches = []
    current_lines = []
    current_bytes = 0
    for event_index, event in enumerate(events, start=1):
        line = _serialize_event(event)
        if len(line) > _MAX_REQUEST_BYTES:
            raise NonRetryableSecOpsError(
                f"Event {event_index} is {len(line)} bytes, exceeding the 4 MB webhook limit."
            )
        line_bytes = len(line) + (1 if current_lines else 0)
        if current_lines and (
            len(current_lines) >= batch_size or current_bytes + line_bytes > _MAX_REQUEST_BYTES
        ):
            payload = b"\n".join(current_lines)
            batches.append(
                {
                    "events": len(current_lines),
                    "bytes": len(payload),
                    "payload": payload,
                }
            )
            current_lines = [line]
            current_bytes = len(line)
        else:
            current_lines.append(line)
            current_bytes += line_bytes

    if current_lines:
        payload = b"\n".join(current_lines)
        batches.append(
            {
                "events": len(current_lines),
                "bytes": len(payload),
                "payload": payload,
            }
        )
    return batches


@retry(
    stop=stop_after_attempt(4),
    wait=_secops_wait,
    retry=retry_if_exception_type((requests.RequestException, RetryableSecOpsError)),
    before_sleep=before_sleep_log(_log, logging.WARNING),
    reraise=True,
)
def _post_batch(url: str, headers: dict, payload: bytes, post_fn=requests.post):
    """POST one NDJSON batch to Google SecOps."""
    response = post_fn(url, headers=headers, data=payload, timeout=60)
    if response.status_code in (408, 429) or response.status_code >= 500:
        retry_after = _parse_retry_after(response.headers.get("Retry-After"))
        raise RetryableSecOpsError(
            f"Retryable error {response.status_code}: {response.text[:300]}",
            retry_after=retry_after,
        )
    if response.status_code >= 400:
        raise NonRetryableSecOpsError(
            f"Client error {response.status_code}: {response.text[:500]}\n"
            "  -> 401/403: check webhook URL query auth or fallback headers\n"
            "  -> 413: reduce BATCH_SIZE or trim oversized events"
        )
    return response


def send_to_secops(events: list[dict], config: dict, post_fn=requests.post) -> dict:
    """Send the prepared events to Google SecOps or render a dry-run preview."""
    total = len(events)
    auth_mode = describe_auth_mode(config["SECOPS_WEBHOOK_URL"])
    if total == 0:
        print("No events selected for Google SecOps delivery.")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": 0,
            "planned_bytes": 0,
            "auth_mode": auth_mode,
            "dry_run": config["DRY_RUN"],
            "skipped": True,
            "errors": [],
        }

    batches = _build_event_batches(events, config["BATCH_SIZE"])
    planned_bytes = sum(batch["bytes"] for batch in batches)

    if config["DRY_RUN"]:
        first_batch = batches[0]
        preview = first_batch["payload"][:1500].decode("utf-8", errors="replace")
        print(f"[DRY_RUN] {total} event(s) would be sent to {mask_webhook_url(config['SECOPS_WEBHOOK_URL'])}")
        print(f"Auth mode   : {auth_mode}")
        print(f"Batches     : {len(batches)}")
        print(f"Total bytes : {planned_bytes}")
        print(f"First batch : {first_batch['events']} event(s) / {first_batch['bytes']} bytes")
        print("\nFirst batch preview:")
        print(preview)
        if len(first_batch["payload"]) > 1500:
            print("... <truncated>")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": len(batches),
            "planned_bytes": planned_bytes,
            "auth_mode": auth_mode,
            "dry_run": True,
            "skipped": False,
            "errors": [],
        }

    headers = _build_headers(config)
    sent = 0
    successful_batches = 0
    failed_batches = 0
    failed_event_count = 0
    errors = []
    attempted_batches = 0

    for batch_index, batch in enumerate(batches, start=1):
        attempted_batches = batch_index
        try:
            _post_batch(config["SECOPS_WEBHOOK_URL"], headers, batch["payload"], post_fn=post_fn)
            sent += batch["events"]
            successful_batches += 1
            print(
                f"  Batch {batch_index}: sent {batch['events']} event(s) / {batch['bytes']} bytes "
                f"(total {sent}/{total})"
            )
        except Exception as exc:
            failed_batches += 1
            failed_event_count += batch["events"]
            errors.append(
                {
                    "batch": batch_index,
                    "events": batch["events"],
                    "bytes": batch["bytes"],
                    "error": str(exc),
                }
            )
            print(f"  Batch {batch_index}: failed after retries - {exc}")
            break

    if failed_batches:
        print(f"Stopped after {sent} sent event(s); {failed_event_count} event(s) remain unsent.")
    else:
        print(f"Complete: {sent} event(s) sent in {successful_batches} batch(es).")

    return {
        "sent": sent,
        "batches": successful_batches,
        "attempted_batches": attempted_batches,
        "failed_batches": failed_batches,
        "failed_event_count": failed_event_count,
        "planned_batches": len(batches),
        "planned_bytes": planned_bytes,
        "auth_mode": auth_mode,
        "dry_run": False,
        "skipped": False,
        "errors": errors,
    }


def build_debug_snapshot(config: dict, app_state: dict) -> dict:
    """Build a JSON-safe debug snapshot for the final notebook cell."""
    query_result_snapshot = {}
    for group_key, result in app_state["query_results"].items():
        query_result_snapshot[group_key] = {
            "account_label": result["account_label"],
            "query_key": result["query_key"],
            "query_title": result["query_title"],
            "domain": result["domain"],
            "log_type": result["chronicle_log_type"],
            "status": result["status"],
            "row_count": result["row_count"],
            "columns": list(result.get("columns", [])),
            "error": result.get("error"),
        }

    return {
        "CONFIG": summarize_runtime_config(config),
        "APP_STATE": {
            "selected_accounts": list(app_state["selected_accounts"]),
            "selected_checks": list(app_state["selected_checks"]),
            "checked_groups": list(app_state["checked_groups"]),
            "dry_run": app_state["dry_run"],
            "last_run_selection": json_safe_value(app_state["last_run_selection"]),
            "selection_dirty": app_state["selection_dirty"],
            "dirty_reason": app_state["dirty_reason"],
            "sanity_check_results": json_safe_value(app_state["sanity_check_results"]),
            "header": build_header_snapshot(config, app_state),
            "selected_events": len(app_state["selected_events"]),
        },
        "QUERY_RESULTS": query_result_snapshot,
        "PAYLOAD_PLAN": json_safe_value(app_state["payload_plan"]),
        "SECOPS_RESULT": json_safe_value(app_state["secops_result"]),
    }


if not globals().get("__NOTEBOOK_TEST__", False):
    PROJECT_DIR, ENV_PATH = load_environment()
    CONFIG = build_runtime_config(
        os.environ,
        project_dir=PROJECT_DIR,
        env_path=ENV_PATH,
        prompt_for_missing=True,
    )
    APP_STATE = create_app_state(CONFIG)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    CONFIG_SUMMARY = summarize_runtime_config(CONFIG)
    print(json.dumps(CONFIG_SUMMARY, indent=2))


In [ ]:
# Cell 3 - Unified control panel
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


GROUP_CATALOG = APP_STATE["group_catalog"]
CHECK_LOOKUP = GROUP_CATALOG["check_lookup"]
ACCOUNT_SELECTOR = widgets.SelectMultiple(
    options=[(account["label"], account["label"]) for account in GROUP_CATALOG["accounts"]],
    value=tuple(APP_STATE["selected_accounts"]),
    description="Accounts",
    rows=min(6, max(1, len(GROUP_CATALOG["accounts"]))),
    layout=widgets.Layout(width="320px"),
)
DRY_RUN_TOGGLE = widgets.Checkbox(
    value=APP_STATE["dry_run"],
    description="DRY_RUN",
    indent=False,
)
RUN_BUTTON = widgets.Button(
    description="Run Selected",
    button_style="primary",
    icon="play",
)
SEND_BUTTON = widgets.Button(
    description="Send Selected to SecOps",
    icon="paper-plane",
)
HEADER_HTML = widgets.HTML()
STATUS_HTML = widgets.HTML()
DOMAIN_CARDS_BOX = widgets.VBox()
RUN_TABLE_BOX = widgets.VBox()
CHECKBOX_BY_QUERY = {}
ROW_CHECKBOXES = {}
RUN_OUTPUT = widgets.Output()
PREVIEW_OUTPUT = widgets.Output()
SEND_OUTPUT = widgets.Output()
OUTPUT_TABS = widgets.Tab(children=[RUN_OUTPUT, PREVIEW_OUTPUT, SEND_OUTPUT])
OUTPUT_TABS.set_title(0, "Run Log")
OUTPUT_TABS.set_title(1, "Preview")
OUTPUT_TABS.set_title(2, "SecOps Log")


def make_html_cell(value: str, width: str) -> widgets.HTML:
    """Render one run-table cell."""
    return widgets.HTML(
        value=value,
        layout=widgets.Layout(width=width, overflow="hidden"),
    )


def refresh_state_from_widgets() -> dict:
    """Sync top-level widgets back into APP_STATE."""
    set_selected_accounts(APP_STATE, list(ACCOUNT_SELECTOR.value))
    snapshot = sync_runtime_selection(APP_STATE, APP_STATE["checked_groups"], DRY_RUN_TOGGLE.value)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    return snapshot


def render_header() -> None:
    """Render the top-level status summary."""
    header = build_header_snapshot(CONFIG, APP_STATE)
    mode_label = "DRY_RUN" if header["dry_run"] else "LIVE"
    dirty_label = "yes" if header["selection_dirty"] else "no"
    HEADER_HTML.value = (
        "<div style='border:1px solid #cbd5e1;border-radius:10px;padding:12px;background:#f8fafc'>"
        f"<div><b>Feed</b>: {html.escape(header['masked_webhook_url'])}</div>"
        f"<div style='margin-top:6px'><b>Parser</b>: {html.escape(header['chronicle_parser'])}"
        f" | <b>Transport</b>: {html.escape(header['chronicle_transport'])}"
        f" | <b>Auth</b>: {html.escape(header['auth_mode'])}</div>"
        f"<div style='margin-top:6px'><b>Mode</b>: {mode_label}"
        f" | <b>Accounts</b>: {header['selected_account_count']}"
        f" | <b>Visible rows</b>: {header['visible_group_count']}"
        f" | <b>Selected rows</b>: {header['selected_group_count']}"
        f" | <b>Eligible</b>: {header['eligible_group_count']}"
        f" | <b>Dirty</b>: {dirty_label}</div>"
        "</div>"
    )
    STATUS_HTML.value = (
        "<div style='font-size:12px;color:#475569;margin:6px 0 12px 0'>"
        f"{html.escape(header['send_reason'])}</div>"
    )
    RUN_BUTTON.disabled = header["selected_group_count"] == 0
    SEND_BUTTON.disabled = not header["send_enabled"]


def render_domain_cards() -> None:
    """Render report-driven domain cards."""
    sections = []
    for domain_entry in GROUP_CATALOG["domains"]:
        cards = []
        for check in domain_entry["checks"]:
            checkbox = CHECKBOX_BY_QUERY.get(check["key"])
            if checkbox is None:
                checkbox = widgets.Checkbox(
                    value=True,
                    description=check["title"],
                    indent=False,
                    layout=widgets.Layout(width="auto"),
                )
                checkbox.style.description_width = "initial"
                CHECKBOX_BY_QUERY[check["key"]] = checkbox

                def _on_change(change, query_key=check["key"]):
                    set_check_selection_state(APP_STATE, query_key, bool(change["new"]))
                    refresh_state_from_widgets()
                    render_header()
                    render_run_table()
                    render_preview_output()

                checkbox.observe(_on_change, names="value")

            card_html = widgets.HTML(
                (
                    f"<div style='font-size:12px;color:#334155'>"
                    f"<div><b>Priority</b>: {html.escape(check['priority'])}</div>"
                    f"<div><b>Sources</b>: {html.escape(', '.join(check['sources']))}</div>"
                    f"<div><b>Watermark</b>: {html.escape(check['watermark_column'])}</div>"
                    f"<div><b>Latency</b>: {check['max_latency_minutes']} min</div>"
                    f"<div><b>Cadence</b>: {html.escape(check['recommended_cadence'])}</div>"
                    f"<div><b>Log Type</b>: {html.escape(check['chronicle_log_type'])}</div>"
                    f"<div style='margin-top:6px'>{html.escape(check['description'])}</div>"
                    f"</div>"
                )
            )
            cards.append(
                widgets.VBox(
                    [checkbox, card_html],
                    layout=widgets.Layout(
                        border="1px solid #cbd5e1",
                        border_radius="10px",
                        padding="12px",
                        width="360px",
                        min_height="210px",
                    ),
                )
            )
        sections.append(
            widgets.VBox(
                [
                    widgets.HTML(f"<h3 style='margin:8px 0'>{html.escape(domain_entry['domain'])}</h3>"),
                    widgets.HBox(cards, layout=widgets.Layout(flex_flow="row wrap", gap="12px")),
                ]
            )
        )
    DOMAIN_CARDS_BOX.children = tuple(sections)


def render_run_table() -> None:
    """Render the selectable account/check run table."""
    rows = build_run_table_rows(CONFIG, APP_STATE)
    ROW_CHECKBOXES.clear()
    if not rows:
        RUN_TABLE_BOX.children = (
            widgets.HTML("<div style='color:#64748b'>Select at least one account to see run rows.</div>"),
        )
        return

    header_row = widgets.HBox(
        [
            make_html_cell("<b>Selected</b>", "70px"),
            make_html_cell("<b>Account</b>", "120px"),
            make_html_cell("<b>Check</b>", "220px"),
            make_html_cell("<b>Domain</b>", "150px"),
            make_html_cell("<b>Source</b>", "260px"),
            make_html_cell("<b>Watermark</b>", "140px"),
            make_html_cell("<b>Latency</b>", "90px"),
            make_html_cell("<b>Cadence</b>", "150px"),
            make_html_cell("<b>Log Type</b>", "150px"),
            make_html_cell("<b>Status</b>", "170px"),
            make_html_cell("<b>Rows</b>", "80px"),
        ],
        layout=widgets.Layout(border_bottom="1px solid #cbd5e1", padding="4px 0"),
    )

    rendered_rows = [header_row]
    for row in rows:
        checkbox = widgets.Checkbox(
            value=row["selected"],
            description="",
            indent=False,
            layout=widgets.Layout(width="40px"),
        )
        ROW_CHECKBOXES[row["group_key"]] = checkbox

        def _on_row_change(change, group_key=row["group_key"]):
            set_group_selection_state(APP_STATE, group_key, bool(change["new"]))
            refresh_state_from_widgets()
            render_header()
            render_preview_output()

        checkbox.observe(_on_row_change, names="value")
        result = APP_STATE["query_results"][row["group_key"]]
        rendered_rows.append(
            widgets.HBox(
                [
                    checkbox,
                    make_html_cell(html.escape(row["account"]), "120px"),
                    make_html_cell(html.escape(row["check"]), "220px"),
                    make_html_cell(html.escape(row["domain"]), "150px"),
                    make_html_cell(html.escape(row["source"] or "-"), "260px"),
                    make_html_cell(html.escape(row["watermark"] or "-"), "140px"),
                    make_html_cell(str(row["latency"]), "90px"),
                    make_html_cell(html.escape(row["cadence"]), "150px"),
                    make_html_cell(html.escape(row["log_type"]), "150px"),
                    widgets.HTML(build_status_badge(result), layout=widgets.Layout(width="170px")),
                    make_html_cell(str(row["row_count"]), "80px"),
                ],
                layout=widgets.Layout(border_bottom="1px solid #e2e8f0", padding="4px 0"),
            )
        )

    RUN_TABLE_BOX.children = (
        widgets.Box(
            rendered_rows,
            layout=widgets.Layout(overflow_x="auto", display="flex", flex_flow="column", gap="2px"),
        ),
    )


def render_preview_output() -> None:
    """Render query results and Chronicle payload previews."""
    with PREVIEW_OUTPUT:
        clear_output(wait=True)
        if APP_STATE["last_run_selection"] is None:
            print("Run selected rows to preview results.")
            return

        print(f"Latest run mode: {'DRY_RUN' if APP_STATE['last_run_selection']['dry_run'] else 'LIVE'}")
        print(f"Latest run rows: {len(APP_STATE['last_run_selection']['checked_groups'])}")
        if APP_STATE["selection_dirty"]:
            print(APP_STATE["dirty_reason"])

        preview_rows = []
        for row in build_run_table_rows(CONFIG, APP_STATE):
            result = APP_STATE["query_results"][row["group_key"]]
            preview_rows.append(
                {
                    "account": row["account"],
                    "check": row["check"],
                    "log_type": row["log_type"],
                    "status": result["status"],
                    "rows": result["row_count"],
                    "error": result["error"],
                }
            )
        if preview_rows:
            display(pd.DataFrame(preview_rows))

        print("\nPayload plan:")
        print(json.dumps(json_safe_value(APP_STATE["payload_plan"]), indent=2))

        if APP_STATE["selected_events"]:
            print("\nFirst event preview:")
            print(json.dumps(APP_STATE["selected_events"][0], indent=2))

        for group_key in APP_STATE["checked_groups"]:
            result = APP_STATE["query_results"][group_key]
            if result["status"] != "success" or result["row_count"] == 0:
                continue
            display(
                HTML(
                    (
                        f"<h4>{html.escape(result['account_label'])} / "
                        f"{html.escape(result['query_title'])} "
                        f"[{html.escape(result['chronicle_log_type'])}]</h4>"
                    )
                )
            )
            display(result["dataframe"].head())


def on_accounts_change(_change):
    """Refresh the visible run rows after the account selection changes."""
    refresh_state_from_widgets()
    render_header()
    render_run_table()
    render_preview_output()


def on_dry_run_change(_change):
    """Refresh the header after the dry-run mode changes."""
    refresh_state_from_widgets()
    render_header()
    render_preview_output()


def on_run_clicked(_button):
    """Run the currently selected account/check rows."""
    snapshot = refresh_state_from_widgets()
    render_header()
    render_run_table()

    with RUN_OUTPUT:
        clear_output(wait=True)
        if not snapshot["checked_groups"]:
            print("Select at least one row to run.")
            return
        print(f"Running {len(snapshot['checked_groups'])} selected row(s).")
        print(f"Mode: {'DRY_RUN' if snapshot['dry_run'] else 'LIVE'}")
        print(f"Accounts: {', '.join(APP_STATE['selected_accounts'])}")

    APP_STATE["sanity_check_results"] = []
    APP_STATE["selected_events"] = []
    APP_STATE["payload_plan"] = {
        "selected_groups": [],
        "total_events": 0,
        "generated_at": None,
        "dry_run": snapshot["dry_run"],
    }
    APP_STATE["secops_result"] = None
    APP_STATE["query_results"] = build_initial_query_results(CONFIG, GROUP_CATALOG)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    render_run_table()
    render_preview_output()

    sanity_results = run_selected_account_sanity_checks(CONFIG, APP_STATE["selected_accounts"])
    APP_STATE["sanity_check_results"] = sanity_results
    with RUN_OUTPUT:
        print("\nSanity checks:")
        display(pd.DataFrame(sanity_results))

    def progress_callback(group_key: str, result: dict) -> None:
        APP_STATE["query_results"][group_key] = result
        sync_config_runtime_fields(CONFIG, APP_STATE)
        render_run_table()

    results = execute_selected_groups(
        CONFIG,
        snapshot["checked_groups"],
        progress_fn=progress_callback,
    )
    for group_key, result in results.items():
        APP_STATE["query_results"][group_key] = result

    APP_STATE["last_run_selection"] = snapshot
    APP_STATE["selection_dirty"] = False
    APP_STATE["dirty_reason"] = "Ready to send."
    APP_STATE["selected_events"], APP_STATE["payload_plan"] = build_selected_events(
        APP_STATE["query_results"],
        APP_STATE["checked_groups"],
        dry_run=APP_STATE["dry_run"],
    )
    sync_config_runtime_fields(CONFIG, APP_STATE)

    with RUN_OUTPUT:
        print("\nExecution summary:")
        display(pd.DataFrame(build_run_table_rows(CONFIG, APP_STATE)))
        print(json.dumps(json_safe_value(summarize_query_results(APP_STATE["query_results"])), indent=2))

    render_header()
    render_run_table()
    render_preview_output()


def on_send_clicked(_button):
    """Send the currently selected successful rows to Google SecOps."""
    refresh_state_from_widgets()
    send_state = reduce_send_state(APP_STATE)
    render_header()

    with SEND_OUTPUT:
        clear_output(wait=True)
        if not send_state["enabled"]:
            print(send_state["reason"])
            return

        events, payload_plan = build_selected_events(
            APP_STATE["query_results"],
            APP_STATE["checked_groups"],
            dry_run=APP_STATE["dry_run"],
        )
        APP_STATE["selected_events"] = events
        APP_STATE["payload_plan"] = payload_plan
        sender_config = build_sender_config(CONFIG, APP_STATE)
        sync_config_runtime_fields(CONFIG, APP_STATE)

        print(f"Sending {len(events)} event(s) from {len(payload_plan['selected_groups'])} row(s).")
        print(f"Mode: {'DRY_RUN' if sender_config['DRY_RUN'] else 'LIVE'}")
        print(f"Parser: {CONFIG['CHRONICLE_PARSER']}")
        try:
            result = send_to_secops(events, sender_config)
        except Exception as exc:
            result = {
                "sent": 0,
                "batches": 0,
                "attempted_batches": 0,
                "failed_batches": 1,
                "failed_event_count": len(events),
                "planned_batches": 0,
                "planned_bytes": 0,
                "auth_mode": describe_auth_mode(sender_config["SECOPS_WEBHOOK_URL"]),
                "dry_run": sender_config["DRY_RUN"],
                "skipped": False,
                "errors": [{"error": str(exc)}],
            }
            print(f"Send failed: {exc}")

    APP_STATE["secops_result"] = result
    render_header()
    render_preview_output()


ACCOUNT_SELECTOR.observe(on_accounts_change, names="value")
DRY_RUN_TOGGLE.observe(on_dry_run_change, names="value")
RUN_BUTTON.on_click(on_run_clicked)
SEND_BUTTON.on_click(on_send_clicked)

refresh_state_from_widgets()
render_header()
render_domain_cards()
render_run_table()
render_preview_output()

display(
    widgets.VBox(
        [
            widgets.HTML("<h2>Snowflake to Chronicle Check Console</h2>"),
            HEADER_HTML,
            widgets.HBox(
                [
                    ACCOUNT_SELECTOR,
                    widgets.VBox([DRY_RUN_TOGGLE, RUN_BUTTON, SEND_BUTTON]),
                ],
                layout=widgets.Layout(gap="16px", align_items="flex-start"),
            ),
            STATUS_HTML,
            widgets.HTML("<h3>Check Catalog</h3>"),
            DOMAIN_CARDS_BOX,
            widgets.HTML("<h3>Run Table</h3>"),
            RUN_TABLE_BOX,
            OUTPUT_TABS,
        ]
    )
)


In [ ]:
# Cell 4 - Summary and debug
DEBUG_SNAPSHOT = build_debug_snapshot(CONFIG, APP_STATE)
print(json.dumps(DEBUG_SNAPSHOT, indent=2))


In [ ]:
# Cell 5 — Redesigned control panel
# Streamlined 4-zone UI: account toggles · check toggles · live matrix · action bar
# Requires Cells 1–2 to have been executed (CONFIG, APP_STATE already initialised).

import html as _html
import json

import ipywidgets as widgets
from IPython.display import clear_output, display

# ── Widgets ───────────────────────────────────────────────────────────────────

acct_boxes: dict[str, widgets.Checkbox] = {
    acct["label"]: widgets.Checkbox(
        value=acct["label"] in APP_STATE["selected_accounts"],
        description=acct["label"],
        indent=False,
        layout=widgets.Layout(width="auto"),
    )
    for acct in CONFIG["SNOWFLAKE_ACCOUNTS"]
}

check_boxes: dict[str, widgets.Checkbox] = {
    chk["key"]: widgets.Checkbox(
        value=True,
        description=chk["title"],
        indent=False,
        layout=widgets.Layout(width="240px"),
    )
    for chk in CONFIG["SECURITY_CHECKS"]
}

dry_run_btn = widgets.ToggleButton(
    value=APP_STATE["dry_run"],
    description="DRY RUN",
    button_style="warning",
    icon="eye",
    layout=widgets.Layout(width="130px", height="36px"),
)
run_btn = widgets.Button(
    description="Run Selected",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="140px", height="36px"),
)
send_btn = widgets.Button(
    description="Send to SecOps",
    button_style="success",
    icon="paper-plane",
    disabled=True,
    layout=widgets.Layout(width="160px", height="36px"),
)
status_lbl = widgets.HTML()
matrix_html = widgets.HTML()
run_output = widgets.Output()
preview_output = widgets.Output()
send_output = widgets.Output()
output_tabs = widgets.Tab(children=[run_output, preview_output, send_output])
output_tabs.set_title(0, "Run Log")
output_tabs.set_title(1, "Preview")
output_tabs.set_title(2, "SecOps Log")

# ── Pure read helpers ─────────────────────────────────────────────────────────

def _selected_accounts() -> list[str]:
    return [label for label, cb in acct_boxes.items() if cb.value]


def _selected_checks() -> list[str]:
    return [key for key, cb in check_boxes.items() if cb.value]


def _checked_groups() -> list[str]:
    return [
        make_group_key(label, key)
        for label in _selected_accounts()
        for key in _selected_checks()
    ]


def _cell_html(result: dict) -> str:
    """Return a styled HTML fragment for one matrix cell."""
    status = result.get("status", "pending")
    row_count = result.get("row_count", 0)
    error = result.get("error") or ""
    if status == "success":
        return (
            f"<span style='color:#059669'>&#10003;&nbsp;"
            f"{row_count}&nbsp;row{'s' if row_count != 1 else ''}</span>"
        )
    if status == "empty":
        return "<span style='color:#d97706'>&#8709;&nbsp;empty</span>"
    if status == "error":
        short = _html.escape(error[:60])
        full = _html.escape(error)
        return f"<span style='color:#dc2626' title='{full}'>&#10007;&nbsp;{short}</span>"
    if status == "running":
        return "<span style='color:#3b82f6'>&#9203;&nbsp;running&hellip;</span>"
    return "<span style='color:#94a3b8'>&mdash;</span>"


def _build_matrix_html() -> str:
    """Build an HTML status matrix from APP_STATE['query_results']."""
    accts = _selected_accounts()
    sel_keys = set(_selected_checks())
    checks = [c for c in CONFIG["SECURITY_CHECKS"] if c["key"] in sel_keys]

    if not accts or not checks:
        return (
            "<p style='color:#94a3b8;font-size:13px;margin:10px 0'>"
            "Select at least one account and one check.</p>"
        )

    th = (
        "padding:8px 14px;background:#f1f5f9;border:1px solid #e2e8f0;"
        "font-size:12px;font-weight:600;text-align:left;white-space:nowrap"
    )
    td_label = (
        "padding:8px 14px;border:1px solid #e2e8f0;"
        "font-size:12px;white-space:nowrap"
    )
    td_cell = (
        "padding:8px 14px;border:1px solid #e2e8f0;"
        "font-size:12px;white-space:nowrap;text-align:center"
    )

    rows = [
        "<table style='border-collapse:collapse;margin-top:10px'>",
        "<thead><tr>",
        f"<th style='{th}'>Check</th>",
    ]
    for label in accts:
        rows.append(f"<th style='{th}'>{_html.escape(label)}</th>")
    rows.append("</tr></thead><tbody>")

    for chk in checks:
        log_type = _html.escape(chk.get("chronicle_log_type", ""))
        rows.append(
            f"<tr><td style='{td_label}'>"
            f"<b>{_html.escape(chk['title'])}</b>"
            f"<br><span style='color:#94a3b8;font-size:11px'>{log_type}</span>"
            f"</td>"
        )
        for label in accts:
            gk = make_group_key(label, chk["key"])
            result = APP_STATE["query_results"].get(gk, {})
            rows.append(f"<td style='{td_cell}'>{_cell_html(result)}</td>")
        rows.append("</tr>")

    rows.append("</tbody></table>")
    return "".join(rows)


# ── State sync ────────────────────────────────────────────────────────────────

def _sync_app_state(checked: list[str]) -> dict:
    """Synchronise widget state onto APP_STATE and compute rerun state."""
    APP_STATE["selected_accounts"] = _selected_accounts()
    APP_STATE["selected_checks"] = _selected_checks()
    checked_set = set(checked)
    for gk in APP_STATE["group_catalog"]["group_order"]:
        APP_STATE["group_selection_state"][gk] = gk in checked_set
    snapshot = sync_runtime_selection(APP_STATE, checked, dry_run_btn.value)
    sync_config_runtime_fields(CONFIG, APP_STATE)
    return snapshot


def _render_preview_output() -> None:
    """Render query results and payload previews in the preview tab."""
    with preview_output:
        clear_output(wait=True)
        if APP_STATE["last_run_selection"] is None:
            print("Run selected rows to preview results.")
            return

        last_run = APP_STATE["last_run_selection"]
        mode = "DRY_RUN" if last_run["dry_run"] else "LIVE"
        print(f"Latest run mode: {mode}")
        print(f"Latest run rows: {len(last_run['checked_groups'])}")
        if APP_STATE["selection_dirty"]:
            print(APP_STATE["dirty_reason"])

        preview_rows = []
        for row in build_run_table_rows(CONFIG, APP_STATE):
            result = APP_STATE["query_results"][row["group_key"]]
            preview_rows.append(
                {
                    "account": row["account"],
                    "check": row["check"],
                    "log_type": row["log_type"],
                    "status": result["status"],
                    "rows": result["row_count"],
                    "error": result["error"],
                }
            )
        if preview_rows:
            display(pd.DataFrame(preview_rows))

        print("\nPayload plan:")
        print(json.dumps(json_safe_value(APP_STATE["payload_plan"]), indent=2))

        if APP_STATE["selected_events"]:
            print("\nFirst event preview:")
            print(json.dumps(APP_STATE["selected_events"][0], indent=2))

        for group_key in APP_STATE["checked_groups"]:
            result = APP_STATE["query_results"][group_key]
            if result["status"] != "success" or result["row_count"] == 0:
                continue
            print(
                f"\n{result['account_label']} / {result['query_title']} "
                f"[{result['chronicle_log_type']}]"
            )
            display(result["dataframe"].head())


def _render() -> None:
    """Rebuild all derived UI state from APP_STATE. Single render path."""
    matrix_html.value = _build_matrix_html()
    send_state = reduce_send_state(APP_STATE)
    send_btn.disabled = not send_state["enabled"]
    run_btn.disabled = not bool(_checked_groups())
    status_lbl.value = (
        f"<span style='color:#475569;font-size:12px'>"
        f"{_html.escape(send_state['reason'])}"
        f"</span>"
    )
    _render_preview_output()


# ── Event handlers ────────────────────────────────────────────────────────────

def on_selection_change(_change) -> None:
    _sync_app_state(_checked_groups())
    _render()


def on_dry_run_change(_change) -> None:
    _sync_app_state(_checked_groups())
    _render()


def on_run_clicked(_btn) -> None:
    checked = _checked_groups()
    if not checked:
        output_tabs.selected_index = 0
        with run_output:
            clear_output(wait=True)
            print("Select at least one account and check before running.")
        return

    snapshot = _sync_app_state(checked)
    APP_STATE["query_results"] = build_initial_query_results(
        CONFIG, APP_STATE["group_catalog"]
    )
    APP_STATE["sanity_check_results"] = []
    APP_STATE["selected_events"] = []
    APP_STATE["payload_plan"] = {
        "selected_groups": [],
        "total_events": 0,
        "generated_at": None,
        "dry_run": snapshot["dry_run"],
    }
    APP_STATE["secops_result"] = None
    # Invalidate send until this run finishes cleanly.
    APP_STATE["last_run_selection"] = None
    APP_STATE["selection_dirty"] = True
    APP_STATE["dirty_reason"] = "Run selected rows first."
    sync_config_runtime_fields(CONFIG, APP_STATE)

    output_tabs.selected_index = 0
    with run_output:
        clear_output(wait=True)
        mode = "DRY_RUN" if snapshot["dry_run"] else "LIVE"
        print(f"Running {len(checked)} group(s) \u2014 mode: {mode}")
        print(f"Accounts : {', '.join(_selected_accounts())}")
        print(f"Checks   : {', '.join(_selected_checks())}")

    _render()

    sanity_results = run_selected_account_sanity_checks(CONFIG, APP_STATE["selected_accounts"])
    APP_STATE["sanity_check_results"] = sanity_results
    with run_output:
        print("\nSanity checks:")
        display(pd.DataFrame(sanity_results))

    def _progress(gk: str, result: dict) -> None:
        APP_STATE["query_results"][gk] = result
        sync_config_runtime_fields(CONFIG, APP_STATE)
        _render()

    results = execute_selected_groups(CONFIG, checked, progress_fn=_progress)
    for gk, r in results.items():
        APP_STATE["query_results"][gk] = r

    events, plan = build_selected_events(APP_STATE["query_results"], checked, dry_run=snapshot["dry_run"])
    APP_STATE["selected_events"] = events
    APP_STATE["payload_plan"] = plan
    APP_STATE["last_run_selection"] = get_selection_snapshot(
        checked,
        snapshot["dry_run"],
        APP_STATE["group_catalog"],
    )
    APP_STATE["selection_dirty"] = False
    APP_STATE["dirty_reason"] = "Ready to send."
    sync_config_runtime_fields(CONFIG, APP_STATE)

    with run_output:
        status_icons = {
            "success": "\u2713",
            "error": "\u2717",
            "empty": "\u2205",
            "running": "\u23f3",
            "pending": "\u2014",
        }
        print(f"\nComplete \u2014 {plan['total_events']} event(s) ready.\n")
        print("Execution summary:")
        display(pd.DataFrame(build_run_table_rows(CONFIG, APP_STATE)))
        print(json.dumps(json_safe_value(summarize_query_results(APP_STATE["query_results"])), indent=2))
        for row in build_run_table_rows(CONFIG, APP_STATE):
            icon = status_icons.get(row["status"], "?")
            suffix = f"  \u2014 {row['error'][:80]}" if row["error"] else ""
            print(
                f"  {icon}  {row['account']} / {row['check']}: "
                f"{row['status']} ({row['row_count']} row(s)){suffix}"
            )
        if events:
            print("\nFirst event preview:")
            print(json.dumps(events[0], indent=2))

    _render()


def on_send_clicked(_btn) -> None:
    _sync_app_state(_checked_groups())
    send_state = reduce_send_state(APP_STATE)
    if not send_state["enabled"]:
        output_tabs.selected_index = 2
        with send_output:
            clear_output(wait=True)
            print(send_state["reason"])
        return

    events, plan = build_selected_events(
        APP_STATE["query_results"],
        APP_STATE["checked_groups"],
        dry_run=APP_STATE["dry_run"],
    )
    sender_cfg = build_sender_config(CONFIG, APP_STATE)
    APP_STATE["selected_events"] = events
    APP_STATE["payload_plan"] = plan
    sync_config_runtime_fields(CONFIG, APP_STATE)

    output_tabs.selected_index = 2
    with send_output:
        clear_output(wait=True)
        mode = "DRY_RUN" if sender_cfg["DRY_RUN"] else "LIVE"
        print(
            f"Sending {len(events)} event(s) \u2014 "
            f"mode: {mode}, parser: {CONFIG['CHRONICLE_PARSER']}"
        )
        try:
            result = send_to_secops(events, sender_cfg)
            APP_STATE["secops_result"] = result
            print(json.dumps(result, indent=2))
        except Exception as exc:
            APP_STATE["secops_result"] = {"error": str(exc)}
            print(f"Send failed: {exc}")

    _render()


# ── Observer wiring ───────────────────────────────────────────────────────────

for _cb in [*acct_boxes.values(), *check_boxes.values()]:
    _cb.observe(on_selection_change, names="value")

dry_run_btn.observe(on_dry_run_change, names="value")
run_btn.on_click(on_run_clicked)
send_btn.on_click(on_send_clicked)

# ── Initial render ────────────────────────────────────────────────────────────

_sync_app_state(_checked_groups())
_render()

# ── Layout ────────────────────────────────────────────────────────────────────

_DIVIDER = widgets.HTML(
    "<hr style='border:none;border-top:1px solid #e2e8f0;margin:8px 0'>",
    layout=widgets.Layout(width="100%"),
)

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<h2 style='margin:0 0 2px 0'>Snowflake &rarr; Chronicle</h2>"
            ),
            _DIVIDER,
            widgets.HTML(
                "<span style='font-size:11px;font-weight:600;"
                "letter-spacing:.05em;color:#64748b'>ACCOUNTS</span>"
            ),
            widgets.HBox(
                list(acct_boxes.values()),
                layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
            ),
            widgets.HTML(
                "<span style='font-size:11px;font-weight:600;"
                "letter-spacing:.05em;color:#64748b;margin-top:8px'>CHECKS</span>"
            ),
            widgets.GridBox(
                list(check_boxes.values()),
                layout=widgets.Layout(
                    grid_template_columns="repeat(auto-fill, 240px)",
                    gap="4px",
                ),
            ),
            _DIVIDER,
            matrix_html,
            _DIVIDER,
            widgets.HBox(
                [dry_run_btn, run_btn, send_btn],
                layout=widgets.Layout(gap="8px", align_items="center"),
            ),
            status_lbl,
            output_tabs,
        ],
        layout=widgets.Layout(padding="16px"),
    )
)